In [218]:
import pytesseract
import cv2
import numpy as np
import re
from matplotlib import pyplot as plt
import easyocr

In [219]:
def display(image,cmap='gray',title='title'):
    plt.figure(figsize=(6,6))
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.show()

In [220]:
def deskew(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        angles = [np.arctan2(y2 - y1, x2 - x1) for [[x1, y1, x2, y2]] in lines]
        median_angle = np.median(angles)
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, np.degrees(median_angle), 1.0)
        image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    return image


In [221]:
def preprocess_image(image):
    if len(image.shape) == 3:  
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else: 
        gray = image
    denoised = cv2.fastNlMeansDenoising(gray, None, 30, 7, 21)
    adaptive_thresh = cv2.adaptiveThreshold(
        denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY, 11, 2
    )
    kernel = np.array([[-1, -1, -1], [-1, 9,-1], [-1, -1, -1]])
    sharpened = cv2.filter2D(adaptive_thresh, -1, kernel)
    resized = cv2.resize(sharpened, (0, 0), fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    
    return resized

In [ ]:
def extract_aadhaar_details(image):
    if image is None:
        print("Error: Image not found or loaded correctly.")
        return
    
    image = deskew(image)
    processed_img = preprocess_image(image)
    
    extracted_text = pytesseract.image_to_string(processed_img, lang='eng+hin+ben+guj+pan+kan+mal+tam+tel+ori', config='--psm 6 --oem 1')
    #print("Extracted Text:\n", extracted_text)
    name_match = re.search(r"([A-Z][a-z]+(?: [A-Z][a-z]+)+)", extracted_text)
    gender_match = re.search(r"(MALE|FEMALE|Male|Female)", extracted_text)
    dob_match = re.search(r"(\d{2}/\d{2}/\d{4})", extracted_text)
    aadhaar_match = re.search(r"(\d{4} \d{4} \d{4})", extracted_text)
    
    name = name_match.group(1) if name_match else "Not Found"
    gender = gender_match.group(1) if gender_match else "Not Found"
    dob = dob_match.group(1) if dob_match else "Not Found"
    aadhaar_number = aadhaar_match.group(1) if aadhaar_match else "Not Found"
    
    print(f"Name: {name}")
    print(f"Gender: {gender}")
    print(f"Date of Birth: {dob}")
    print(f"Aadhaar Number: {aadhaar_number}")


In [223]:
img=cv2.imread("aadhar/front/4.jpg")
details = extract_aadhaar_details(img)
print(details)
deskewed=deskew(img)
processed=preprocess_image(deskewed)
#display(deskewed, "gray", "de-skew")
#display(processed, "gray", "preprocessed")

Name: Shraddha Ganesh Nimse
Gender: Female
Date of Birth: 28/08/1989
Aadhaar Number: 3567 7579 0848
None
